In [1]:
import cv2
from PIL import Image
from transformers import pipeline
from collections import Counter
import numpy as np

In [2]:
pipe = pipeline("image-classification",model="Luwayy/disaster_images_model")
print("Model Loaded Successfully")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Model Loaded Successfully


In [3]:
def extract_frames(video_path, fps=1):
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_rate = cap.get(cv2.CAP_PROP_FPS)
    interval = int(frame_rate * fps)
    count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % interval == 0:
            frames.append(frame)
        count += 1
    cap.release()
    print(f"Extracted {len(frames)} frames")
    return frames

In [4]:
def preprocess_for_hf(frame):
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return Image.fromarray(frame)

In [11]:
from collections import defaultdict
def predict_video_advanced(frames, name="Video"):
    labels = []
    scores = []
    all_results = []

    print(f"\n--- {name} Frame Predictions ---")

    for i, frame in enumerate(frames):
        # Convert frame
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb)

        result = pipe(pil_image)   # full list for top-3
        top1 = result[0]

        label = top1["label"]
        score = top1["score"]

        labels.append(label)
        scores.append(score)
        all_results.append(result)

        print(f"Frame {i}: {label} ({score:.2f})")

    # ===============================
    # ✅ FINAL PREDICTION
    # ===============================
    final_label = Counter(labels).most_common(1)[0][0]
    avg_score = sum(scores) / len(scores)

    print("\n--- FINAL RESULT ---")
    print(f"Disaster Type: {final_label}")
    print(f"Confidence: {avg_score:.2f}")

    # ===============================
    # 🚀 SEVERITY CALCULATION
    # ===============================
    if avg_score > 0.85:
        severity = "High"
    elif avg_score > 0.6:
        severity = "Medium"
    else:
        severity = "Low"

    print(f"⚠️ Severity: {severity}")

    # ===============================
    # 🚀 TREND DETECTION
    # ===============================
    if len(scores) > 1:
        if scores[-1] > scores[0]:
            trend = "Increasing"
            print("⚠️ Severity Increasing Over Time")
        else:
            trend = "Stable/Decreasing"
            print("✅ Stable or Decreasing")
    else:
        trend = "Unknown"

    # ===============================
    # 🚀 TOP-3 PREDICTIONS
    # ===============================
    avg_scores = defaultdict(float)

    for res in all_results:
        for item in res:
            label = item['label']
            score = item['score']
            avg_scores[label] += score

    # normalize
    for k in avg_scores:
        avg_scores[k] /= len(all_results)

    top3 = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)[:3]

    print("\n🔥 Top 3 Predictions:")
    for label, score in top3:
        print(f"{label}: {score:.2f}")

    # ===============================
    # 📦 RETURN EVERYTHING (for API)
    # ===============================
    return {
        "final_label": final_label,
        "confidence": avg_score,
        "severity": severity,
        "trend": trend,
        "top3": top3}

In [12]:
test_frames = extract_frames("test.mp4")
result = predict_video_advanced(test_frames, "Test 1")

Extracted 16 frames

--- Test 1 Frame Predictions ---
Frame 0: Fire_Disaster (0.91)
Frame 1: Fire_Disaster (0.91)
Frame 2: Fire_Disaster (0.91)
Frame 3: Fire_Disaster (0.91)
Frame 4: Fire_Disaster (0.92)
Frame 5: Fire_Disaster (0.91)
Frame 6: Fire_Disaster (0.91)
Frame 7: Fire_Disaster (0.91)
Frame 8: Fire_Disaster (0.91)
Frame 9: Fire_Disaster (0.91)
Frame 10: Fire_Disaster (0.91)
Frame 11: Fire_Disaster (0.91)
Frame 12: Fire_Disaster (0.91)
Frame 13: Fire_Disaster (0.91)
Frame 14: Fire_Disaster (0.92)
Frame 15: Fire_Disaster (0.92)

--- FINAL RESULT ---
Disaster Type: Fire_Disaster
Confidence: 0.91
⚠️ Severity: High
⚠️ Severity Increasing Over Time

🔥 Top 3 Predictions:
Fire_Disaster: 0.91
Human_Damage: 0.02
Land_Disaster: 0.02


In [13]:
test_frames = extract_frames("test2.mp4")
result = predict_video_advanced(test_frames, "Test 1")

Extracted 6 frames

--- Test 1 Frame Predictions ---
Frame 0: Water_Disaster (0.91)
Frame 1: Water_Disaster (0.91)
Frame 2: Water_Disaster (0.89)
Frame 3: Water_Disaster (0.90)
Frame 4: Water_Disaster (0.91)
Frame 5: Water_Disaster (0.91)

--- FINAL RESULT ---
Disaster Type: Water_Disaster
Confidence: 0.91
⚠️ Severity: High
⚠️ Severity Increasing Over Time

🔥 Top 3 Predictions:
Water_Disaster: 0.91
Land_Disaster: 0.03
Human_Damage: 0.02


In [14]:
pipe.model.save_pretrained("disaster_model")
pipe.image_processor.save_pretrained("disaster_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['disaster_model\\preprocessor_config.json']